# Next Greater Element II

# Problem Statement

Given a **circular integer array** `nums`, return the next greater element for every element.

The next greater element of an element `x` is the first element greater than `x` that appears after it while moving to the right.

Because the array is circular, after reaching the last element, we continue from the beginning.

If no greater element exists, return:

```text
-1
```

### Input

An integer array:

```text
nums
```

### Output

An array where each position contains the next greater element.

### Examples

```text
Input:
[1, 2, 1]

Output:
[2, -1, 2]
```

Explanation:

For the first `1`:

```text
1 → 2
```

For `2`:

```text
2 → -1
```

There is no greater element anywhere in the circular traversal.

For the last `1`:

```text
1 → 2
```

The `2` is found by wrapping around to the beginning.

---

### Another Example

```text
Input:
[3, 8, 4, 1, 2]

Output:
[8, -1, 8, 2, 3]
```

# Problem Explanation

This is a variation of **Next Greater Element I**.

The normal problem asks:

```text
Find the first greater element to the RIGHT.
```

The difference here is:

```text
The array is CIRCULAR.
```

Consider:

```text
[1, 2, 1]
```

For the last `1`:

```text
1
```

there is nothing greater to its right inside the normal array.

But because the array is circular, we continue from the beginning:

```text
1 → 2 → 1
    ↑
```

So:

```text
1 → 2
```

The main challenge is therefore:

```text
How do we simulate wrapping around
without actually creating a second array?
```

The answer is to process the array as if it were:

```text
nums + nums
```

while using the original indices.

# Input

```python
nums = [1, 2, 1]
```

# Output

```text
[2, -1, 2]
```

# Brute Force

For every element, move to the right and keep checking elements.

Because the array is circular, after reaching the last index, continue from index `0`.

For:

```text
[1, 2, 1]
```

For the last `1`:

```text
index 2 → index 0
```

Then:

```text
nums[0] = 1
nums[1] = 2
```

Since:

```text
2 > 1
```

the answer is:

```text
2
```

In the worst case, each element may inspect almost every other element.

### Complexity

```text
Time  → O(N²)
Space → O(1)
```

We can improve this to O(N) using a Monotonic Stack.

# Key Observation — Circular Array

A circular array:

```text
[1, 2, 3, 4]
```

can be viewed conceptually as:

```text
[1, 2, 3, 4, 1, 2, 3, 4]
```

We do not actually need to create this second array.

Instead, use:

```python
i % n
```

where:

```text
n = len(nums)
```

This converts an index from the doubled range back into an original index.

For example:

```text
i = 0 → 0 % 4 = 0
i = 1 → 1 % 4 = 1
i = 2 → 2 % 4 = 2
i = 3 → 3 % 4 = 3
i = 4 → 4 % 4 = 0
i = 5 → 5 % 4 = 1
```

Therefore:

```text
0 1 2 3 4 5 6 7
↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓
0 1 2 3 0 1 2 3
```

This lets us simulate two passes over the circular array.

# Optimal Approach

Use a **Monotonic Stack** and process the array twice.

Conceptually:

```text
nums + nums
```

For each position:

```text
i = 0 → 2N - 1
```

convert it back to an original index:

```python
index = i % N
```

We maintain a Stack of indices whose next greater element has not yet been found.

### Important Rule

Only the first `N` positions need answers.

The second pass exists only to provide possible greater elements for positions near the end of the original array.

Therefore:

```text
First N positions → calculate answers
Second N positions → help resolve remaining elements
```

# Algorithm

```text
Create result array filled with -1

Create empty Stack

For i from 0 to 2N - 1:

    index = i % N

    While Stack is not empty
    and nums[index] > nums[Stack.top]:

        previous_index = pop()

        result[previous_index] = nums[index]

    If i < N:

        Push index
```

Why only push during the first pass?

Because every original element should be added to the Stack only once.

The second pass is used only to resolve elements that were still waiting.

In [1]:
class Solution:

    def nextGreaterElements(self, nums: list[int]) -> list[int]:

        n = len(nums)

        result = [-1] * n
        stack = []

        for i in range(2 * n):

            index = i % n

            while stack and nums[index] > nums[stack[-1]]:

                previous_index = stack.pop()

                result[previous_index] = nums[index]

            if i < n:
                stack.append(index)

        return result

# Dry Run

Input:

```text
nums = [1, 2, 1]
```

Therefore:

```text
N = 3
```

We conceptually process:

```text
[1, 2, 1, 1, 2, 1]
```

Start:

```text
Result = [-1, -1, -1]
Stack  = []
```

### i = 0

```text
index = 0 % 3 = 0
value = 1
```

Push index `0`:

```text
Stack = [0]
```

---

### i = 1

```text
index = 1
value = 2
```

Compare:

```text
2 > 1
```

Yes.

Resolve index `0`:

```text
Result[0] = 2
```

Stack:

```text
[]
```

Push index `1`:

```text
Stack = [1]
```

---

### i = 2

```text
index = 2
value = 1
```

Compare:

```text
1 > 2
```

False.

Push index `2`:

```text
Stack = [1, 2]
```

---

### i = 3

Now we wrap around.

```text
index = 3 % 3 = 0
value = 1
```

Compare with Stack top:

```text
1 > 1
```

False.

We do not push again because:

```text
i >= N
```

---

### i = 4

```text
index = 4 % 3 = 1
value = 2
```

Compare with Stack top:

```text
2 > 1
```

Yes.

Resolve index `2`:

```text
Result[2] = 2
```

Stack:

```text
[1]
```

Compare again:

```text
2 > 2
```

False.

Final:

```text
Result = [2, -1, 2]
```

Index `1` remains unresolved because no value greater than `2` exists.

Therefore:

```text
Output = [2, -1, 2]
```

# Visualizing the Circular Array

Original:

```text
        ┌───────────────┐
        ↓               │
[1] → [2] → [1] ────────┘
```

The last `1` can continue back to the first element:

```text
last 1
  ↓
first 1
  ↓
2
```

So its next greater element is:

```text
2
```

Conceptually:

```text
Original:

[1, 2, 1]

Circular traversal:

[1, 2, 1, 1, 2, 1]
```

We use the second copy only to discover greater elements for elements in the first copy.

# Edge Cases

### Single Element

```text
Input:
[5]

Output:
[-1]
```

An element cannot be greater than itself.

---

### Increasing Array

```text
Input:
[1, 2, 3, 4]
```

Answers:

```text
[2, 3, 4, -1]
```

Even though the array is circular, `4` has no greater element anywhere.

---

### Decreasing Array

```text
Input:
[4, 3, 2, 1]
```

Because the array is circular:

```text
4 → -1
3 → 4
2 → 4
1 → 4
```

Output:

```text
[-1, 4, 4, 4]
```

---

### All Equal

```text
Input:
[5, 5, 5]
```

Equal is not greater.

```text
Output:
[-1, -1, -1]
```

---

### Circular Match

```text
Input:
[2, 1, 2]
```

For the middle `1`:

```text
1 → 2
```

For the last `2`, there is no greater value.

Output:

```text
[-1, 2, -1]
```

# Complexity

Let:

```text
N = len(nums)
```

We process:

```text
2N
```

positions.

Therefore:

```text
2N = O(N)
```

Each index is pushed at most once and popped at most once.

Therefore:

```text
Time → O(N)
```

The Stack contains at most `N` indices.

The result array also contains `N` elements.

Therefore:

```text
Space → O(N)
```

# Comparison

| Approach | Time | Extra Space |
|---|---:|---:|
| Brute Force | O(N²) | O(1) |
| Doubled Array + Stack | O(N) | O(N) |
| Modulo + Stack | O(N) | O(N) |

The modulo approach avoids physically creating a second copy of the array while still simulating circular traversal.

# Pattern Recognition

This problem combines two important patterns:

### Pattern 1 — Next Greater Element

```text
Next greater
      ↓
Monotonic Stack
```

### Pattern 2 — Circular Array

```text
Need to wrap around
      ↓
Process approximately 2N positions
      ↓
Use i % N
```

Together:

```text
Circular
   +
Next Greater
   ↓
Monotonic Stack
+
Modulo / Two Passes
```

This pattern appears in many interview problems involving circular arrays.

# Takeaway

The main difficulty is not the Monotonic Stack itself.

It is recognizing that a circular array can be simulated as:

```text
nums + nums
```

without actually creating the second array.

Use:

```python
index = i % n
```

and process:

```python
range(2 * n)
```

The complete pattern is:

```text
Circular Array
      ↓
Simulate two passes
      ↓
Monotonic Stack
      ↓
Resolve next greater elements
```

Complexity:

```text
Time  → O(N)
Space → O(N)
```

The key interview trigger is:

```text
Next Greater + Circular Array
                ↓
        Monotonic Stack
        + modulo indexing
```